# 03 – Skill extraction

Goal: turn free-text job descriptions into a clean list of skills per posting.

1. Build a skill dictionary (canonical names + aliases).
2. Extract skills with the dictionary (MVP).
3. Evaluate against gsearch `description_tokens` (precision / recall).

In [1]:
import ast
import pandas as pd

pd.set_option("display.max_rows", 200)

gs = pd.read_csv("../data/raw/gsearch/gsearch_jobs.csv")
gs = gs.drop_duplicates(["title", "company_name", "location"]).copy()

gs["tokens"] = gs["description_tokens"].fillna("[]").apply(ast.literal_eval)
print("postings:", len(gs))
print("postings without any skill:", round((gs["tokens"].str.len() == 0).mean(), 3))
gs["tokens"].str.len().describe().round(1)

postings: 41719
postings without any skill: 0.23


count    41719.0
mean         3.2
std          3.0
min          0.0
25%          1.0
50%          3.0
75%          5.0
max         25.0
Name: tokens, dtype: float64

In [2]:
vocab = gs["tokens"].explode().value_counts()
print("unique skills:", len(vocab))
vocab

unique skills: 133


tokens
sql                      20065
excel                    12882
python                   12320
power_bi                 11006
tableau                  10799
r                         7352
sas                       3365
powerpoint                2962
word                      2921
azure                     2596
aws                       2476
snowflake                 2411
spark                     2056
looker                    1759
go                        1587
spreadsheet               1513
sap                       1441
scala                     1328
spss                      1174
jira                      1168
c                         1142
java                      1135
hadoop                    1052
alteryx                   1049
outlook                    977
nosql                      925
sharepoint                 901
javascript                 883
vba                        877
gcp                        847
bigquery                   792
redshift                   778
q

In [3]:
import re

def show_context(skill, pattern, n=6, width=60):
    """Print n random snippets where the skill's pattern appears."""
    sub = gs[gs["tokens"].apply(lambda t: skill in t)]
    for d in sub["description"].sample(n, random_state=3):
        m = re.search(pattern, d, flags=re.IGNORECASE)
        if m:
            print("...", d[max(0, m.start() - width): m.end() + width].replace("\n", " "), "...")

show_context("go", r"\bgo\b")

... m. Paper resumes will not be accepted. All job seekers must go to the web site to be considered for positions. If you are  ...
... years or over.  Working on this project will require you to go through a standard recruitment process (including passing a ...
... is position.  Please note: Candidate(s) will be required to go through pre-employment screening.  Unissant, Inc. is a prou ...
... ctional teams  Features and Benefits while On Contract:  We go beyond the basic staffing agency offerings You can see the  ...
... ust apply for open positions through their Workday account. Go to Careers app on WD home screen>Click on 'Find Jobs-Intern ...
... Revenue Operations Remote - USA ... The HubSpot Go To Market team uses data from a wide variety of sources to  ...


In [4]:
import re

def show_context(skill, pattern, n=6, width=60):
    """Print n random snippets where the skill's pattern appears."""
    sub = gs[gs["tokens"].apply(lambda t: skill in t)]
    for d in sub["description"].sample(n, random_state=3):
        m = re.search(pattern, d, flags=re.IGNORECASE)
        if m:
            print("...", d[max(0, m.start() - width): m.end() + width].replace("\n", " "), "...")

show_context("c", r"\bc\b")

... perty Intelligence & Risk Management Platform leading the P&C insurance companies depend on to identify and mitigate risk ...
... logy Supervisor II - 365007 $93,396 - $125,136 Annual Range C... View Job ShareThis Copy and Paste #J-18808-Ljbffr ...
... experience, especially data-focused systems • Proficient in C# and Powershell • Good orientation in Azure services • Know ...
... , we trust our associates to innovate the future of s upply c hain & transportation !  As a Transportation Analyst , you' ...
... is, data mapping, and data modeling • Experience with .Net, C#, Aspx, or Javascript  Compensation: • Hourly W2 pay rate $ ...
... f a Quantitative methods worksheet with 3 sections A, B and C to be competed includes questions that needs to be answered ...


### gsearch skill vocabulary

- 133 skills, median 3 per posting, 23% of postings have none.
- It is a silver standard, not gold: "go" (1,587) is almost always the English verb, "c" (1,142) mostly noise (P&C, "A, B and C", C#).
- Duplicate names: postgres/postgresql, mongo/mongodb, js/javascript, node/node.js ...
- Missing: modern data / ML / GenAI tools (dbt, databricks, kafka, mlflow, xgboost, langchain, hugging face ...) and concepts (machine learning, statistics, nlp, a/b testing).

In [5]:
import sys
sys.path.append("..")
from src.skills import extract_skills, SKILLS

print("skills in dictionary:", len(SKILLS))

tests = {
    "Experience with Python, R, and SQL": {"python", "r", "sql"},
    "Our R&D team is growing": set(),
    "Please go to our website to apply": set(),
    "Languages: Java, Go, Rust": {"java", "go", "rust"},
    "We work in the P&C insurance space": set(),
    "Proficient in C, C++ and C#": {"c", "c++", "c#"},
    "You will excel in a fast-paced environment": set(),
    "Advanced Excel and Power BI skills": {"excel", "power bi"},
    "Build RAG pipelines with LangChain and OpenAI": {"rag", "langchain", "openai"},
}

for text, expected in tests.items():
    got = extract_skills(text)
    print("OK " if got == expected else "BAD", text, "→", sorted(got))

skills in dictionary: 88
OK  Experience with Python, R, and SQL → ['python', 'r', 'sql']
OK  Our R&D team is growing → []
OK  Please go to our website to apply → []
OK  Languages: Java, Go, Rust → ['go', 'java', 'rust']
OK  We work in the P&C insurance space → []
OK  Proficient in C, C++ and C# → ['c', 'c#', 'c++']
OK  You will excel in a fast-paced environment → []
OK  Advanced Excel and Power BI skills → ['excel', 'power bi']
OK  Build RAG pipelines with LangChain and OpenAI → ['langchain', 'openai', 'rag']


In [6]:
jobs = pd.read_parquet("../data/processed/linkedin_data_jobs.parquet")
jobs["skills"] = jobs["description"].apply(extract_skills)
jobs["n_skills"] = jobs["skills"].str.len()

print("postings:", len(jobs))
print("postings without any skill:", round((jobs["n_skills"] == 0).mean(), 3))
jobs["n_skills"].describe().round(1)

postings: 2912
postings without any skill: 0.115


count    2912.0
mean        5.5
std         4.5
min         0.0
25%         2.0
50%         5.0
75%         8.0
max        36.0
Name: n_skills, dtype: float64

In [7]:
exploded = jobs[["role", "skills"]].explode("skills").dropna(subset=["skills"])

skill_share = (pd.crosstab(exploded["skills"], exploded["role"])
                 .div(jobs["role"].value_counts(), axis=1))

roles = ["Data Analyst", "BI Analyst", "Data Engineer", "Data Scientist", "ML/AI Engineer"]
skill_share[roles].round(2).sort_values("ML/AI Engineer", ascending=False).head(25)

role,Data Analyst,BI Analyst,Data Engineer,Data Scientist,ML/AI Engineer
skills,,,,,
machine learning,0.12,0.07,0.21,0.68,0.88
python,0.31,0.26,0.55,0.82,0.71
deep learning,0.00,0.00,0.01,0.19,0.41
pytorch,0.00,0.00,0.00,0.11,0.35
nlp,0.01,0.00,0.01,0.19,0.34
generative ai,0.01,0.02,0.01,0.09,0.34
tensorflow,0.00,0.00,0.01,0.09,0.33
llm,0.01,0.00,0.01,0.12,0.32
aws,0.06,0.10,0.41,0.24,0.25


### Skill profile per role (LinkedIn 2023–24, share of postings)

- Data Analyst: sql 52%, python 31%, statistics 31%. BI Analyst: sql 71%, azure 20%.
- Data Engineer: sql 63%, python 55%, aws 41%, azure 37%, spark 32% → cloud + big data.
- Data Scientist: python 82%, statistics 77%, machine learning 68%, r 50% → statistics is what separates DS from MLE (22%).
- ML/AI Engineer: machine learning 88%, python 71%, deep learning 41%, pytorch 35% (> tensorflow 33%), generative ai 34%, llm 32%; engineering tools (c++, kubernetes, go) mostly only here.
- Caveats: mentions ≠ requirements (company intros count too), near-duplicate postings inflate counts, data is from 2023–24.